# ДЗ 7. Часові ряди та напівструктуровані дані: NYC Taxi + JSONB event-log

**Time-series:** NYC TLC Yellow Taxi Trip Records, офіційні Parquet-файли (https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), **два місяці: 2024-02 і 2024-03**, повністю. Якщо TLC недоступний, автоматично вмикається **stub-schema fallback** (60 днів × 5 зон).
**JSONB:** синтетичний детермінований event-log `t7_hw_events` (50 000 подій, 5 типів з різними схемами payload).

Чому два місяці, а не один. Ознака `trips_lag_30d` потребує 30 днів історії, а з одним місяцем (31 день) фільтр `trips_lag_30d IS NOT NULL` залишає **1 день на зону**: training set вироджується. Два місяці дають ~30 повноцінних днів на зону.

Два відхилення від шаблону, обидва пов'язані з коректністю ознак:
1. **Calendar spine.** `t7_hw_taxi_daily` містить кожну пару «зона × день» періоду, включно з днями без поїздок (`trip_count = 0`). Без цього `LAG(trip_count, 7)` бере значення «7 **рядків** тому», а не «7 **днів** тому» (розділ 2.2).
2. **Завантаження через `COPY`** замість `to_sql`: ~6 млн рядків вантажаться за секунди, а не хвилини. Очищення робиться SQL-запитами зі звітом, скільки рядків відкинуло кожне правило.

## 1. Setup і завантаження даних

### 1.1. PostgreSQL через pgserver

In [1]:
# Install dependencies. PyPI ships pgserver wheels only for CPython 3.9-3.12, but its C extension
# is built against the stable ABI (abi3), so on Python 3.13+ we download the cp312 wheel and
# retag it as cp312-abi3, which pip accepts for any CPython >= 3.12.
import subprocess
import sys
from pathlib import Path

PIP = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(PIP + ["psycopg2-binary", "sqlalchemy", "pandas", "pyarrow", "requests"], check=True)

if subprocess.run(PIP + ["pgserver"], capture_output=True).returncode != 0:
    wheel_dir = Path("/tmp/pgserver_wheel")
    subprocess.run(
        [sys.executable, "-m", "pip", "download", "-q", "pgserver==0.1.4",
         "--no-deps", "--only-binary=:all:", "--python-version", "3.12",
         "--platform", "manylinux2014_x86_64", "-d", str(wheel_dir)],
        check=True,
    )
    src = next(wheel_dir.glob("pgserver-0.1.4-cp312-cp312-*.whl"))
    dst = src.with_name(src.name.replace("cp312-cp312", "cp312-abi3"))
    src.rename(dst)
    subprocess.run(PIP + [str(dst)], check=True)

print(f"Python {sys.version.split()[0]}: залежності встановлено")

Python 3.13.18: залежності встановлено


In [2]:
import io
import os
import re
import time
from datetime import date
from urllib.request import urlretrieve

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)

pg = pgserver.get_server("/tmp/hw7_pg", cleanup_mode="stop")
# Pin psycopg2 explicitly: SQLAlchemy 2.1 maps "postgresql://" to psycopg 3, which is not installed
engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1), future=True)


def q(sql: str) -> pd.DataFrame:
    # Run a statement that returns rows and commit
    with engine.begin() as conn:
        return pd.read_sql(text(sql), conn)


def _raw_execute(sql: str, autocommit: bool = False, fetch: bool = False):
    # Execute on a raw psycopg2 cursor without parameters, so "%" (e.g. modulo) is never
    # treated as a placeholder; SQLAlchemy 2.1 exec_driver_sql passes an empty params dict
    raw_conn = engine.raw_connection()
    dbapi_conn = raw_conn.driver_connection  # the real psycopg2 connection behind the pool proxy
    try:
        # Setting autocommit on the pool proxy would be silently ignored, and ANALYZE / VACUUM
        # would then run inside a transaction that the pool rolls back on return
        dbapi_conn.autocommit = autocommit
        with raw_conn.cursor() as cur:
            cur.execute(sql)
            rows = cur.fetchall() if fetch else None
        if not autocommit:
            raw_conn.commit()
        return rows
    finally:
        dbapi_conn.autocommit = False
        raw_conn.close()


def run_sql(sql: str) -> None:
    # Run a multi-statement SQL script in one transaction
    _raw_execute(sql)


def run_autocommit(sql: str) -> None:
    # Run a statement outside a transaction block (ANALYZE / VACUUM)
    _raw_execute(sql, autocommit=True)


def explain(sql: str, show: bool = True) -> tuple[str, float]:
    # Run EXPLAIN (ANALYZE, BUFFERS), print the plan, return plan text and execution time (ms)
    rows = _raw_execute(f"EXPLAIN (ANALYZE, BUFFERS) {sql}", fetch=True)
    plan = "\n".join(r[0] for r in rows)
    if show:
        print(plan)
    return plan, float(re.search(r"Execution Time: ([\d.]+) ms", plan).group(1))


with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

/usr/local/lib/python3.13/dist-packages/platformdirs/api.py:459: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


### 1.2. Варіант A: офіційні Parquet-файли TLC

Файли кешуються в `data/`, тож повторний Run All не завантажує їх знову. Якщо завантаження не вдалося, вмикається варіант B (stub).

In [3]:
TAXI_MONTHS = os.environ.get("HW7_TAXI_MONTHS", "2024-02,2024-03").split(",")
FORCE_STUB = os.environ.get("HW7_FORCE_STUB") == "1"
URL_TEMPLATE = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month}.parquet"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

parquet_paths = {}
USE_STUB = FORCE_STUB
if not USE_STUB:
    try:
        for month in TAXI_MONTHS:
            path = DATA_DIR / f"yellow_tripdata_{month}.parquet"
            if not path.exists():
                urlretrieve(URL_TEMPLATE.format(month=month), path)
            parquet_paths[month] = path
    except Exception as exc:
        print(f"TLC недоступний ({type(exc).__name__}: {exc}); вмикаю stub-schema fallback")
        USE_STUB = True

# Analysis period = the loaded calendar months; records outside it are TLC data errors
if USE_STUB:
    PERIOD_START, PERIOD_END = date(2024, 2, 1), date(2024, 4, 1)
else:
    first_y, first_m = map(int, TAXI_MONTHS[0].split("-"))
    last_y, last_m = map(int, TAXI_MONTHS[-1].split("-"))
    PERIOD_START = date(first_y, first_m, 1)
    PERIOD_END = date(last_y + (last_m == 12), last_m % 12 + 1, 1)

print("Режим:", "STUB" if USE_STUB else f"TLC {', '.join(TAXI_MONTHS)}")
print(f"Період: [{PERIOD_START}, {PERIOD_END})")
for month, path in parquet_paths.items():
    print(f"  {path}  {path.stat().st_size / 1e6:.1f} MB")

Режим: TLC 2024-02, 2024-03
Період: [2024-02-01, 2024-04-01)
  data/yellow_tripdata_2024-02.parquet  50.3 MB
  data/yellow_tripdata_2024-03.parquet  60.1 MB


Raw-шар `nyc_taxi_raw` зберігає **всі** рядки файлів без очищення, з типами Parquet (`TIMESTAMP` без зони, бо в TLC це локальний NYC wall-clock час). Очищення виконується в SQL, тож видно, скільки рядків відкинуло кожне правило. Дані йдуть через `COPY FROM STDIN` помісячно, щоб не тримати весь CSV у пам'яті.

In [4]:
COLUMNS = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID",
    "trip_distance", "fare_amount", "passenger_count",
]

run_sql("""
DROP VIEW IF EXISTS v_t7_hw_taxi_features;
DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;
DROP TABLE IF EXISTS t7_hw_taxi_daily_sparse;
DROP TABLE IF EXISTS t7_hw_taxi_daily;
DROP TABLE IF EXISTS nyc_taxi_raw;
""")

if not USE_STUB:
    run_sql("""
    CREATE TABLE nyc_taxi_raw (
        source_month          TEXT NOT NULL,
        tpep_pickup_datetime  TIMESTAMP,
        tpep_dropoff_datetime TIMESTAMP,
        "PULocationID"        INTEGER,
        "DOLocationID"        INTEGER,
        trip_distance         DOUBLE PRECISION,
        fare_amount           DOUBLE PRECISION,
        passenger_count       DOUBLE PRECISION
    );
    """)
    copy_sql = f"""COPY nyc_taxi_raw (source_month, tpep_pickup_datetime, tpep_dropoff_datetime,
        "PULocationID", "DOLocationID", trip_distance, fare_amount, passenger_count)
        FROM STDIN WITH (FORMAT csv, NULL '')"""
    for month, path in parquet_paths.items():
        t0 = time.perf_counter()
        df = pd.read_parquet(path, columns=COLUMNS)
        for col in ("PULocationID", "DOLocationID"):
            df[col] = df[col].astype("Int64")  # keep integers integer even if NaN is present
        df.insert(0, "source_month", month)
        buffer = io.StringIO()
        df.to_csv(buffer, index=False, header=False, date_format="%Y-%m-%d %H:%M:%S")
        buffer.seek(0)
        raw_conn = engine.raw_connection()
        try:
            with raw_conn.cursor() as cur:
                cur.copy_expert(copy_sql, buffer)
            raw_conn.commit()
        finally:
            raw_conn.close()
        print(f"{month}: {len(df):,} рядків за {time.perf_counter() - t0:.1f} с")
        del df, buffer
    run_autocommit("ANALYZE nyc_taxi_raw")
    display(q("SELECT source_month, COUNT(*) AS n_rows FROM nyc_taxi_raw GROUP BY source_month ORDER BY 1"))

2024-02: 3,007,526 рядків за 47.0 с
2024-03: 3,582,628 рядків за 47.2 с


,source_month,n_rows
0,2024-02,3007526
1,2024-03,3582628


### 1.3. Аудит якості перед агрегацією

Правила очищення з умови (`0 < fare < 500`, `0 < distance < 100`, `PULocationID IS NOT NULL`) плюс межі періоду. TLC-файли містять поодинокі записи з чужих дат (інших місяців і навіть років) — це помилки годинника таксометра, і в денний ряд вони потрапити не повинні.

In [5]:
if not USE_STUB:
    dq = q(f"""
    SELECT
        COUNT(*)                                                                   AS total_rows,
        COUNT(*) FILTER (WHERE tpep_pickup_datetime <  DATE '{PERIOD_START}'
                            OR tpep_pickup_datetime >= DATE '{PERIOD_END}')        AS outside_period,
        MIN(tpep_pickup_datetime)                                                  AS min_pickup,
        MAX(tpep_pickup_datetime)                                                  AS max_pickup,
        COUNT(*) FILTER (WHERE fare_amount IS NULL OR fare_amount <= 0)            AS fare_le_0,
        COUNT(*) FILTER (WHERE fare_amount >= 500)                                 AS fare_ge_500,
        COUNT(*) FILTER (WHERE trip_distance IS NULL OR trip_distance <= 0)        AS distance_le_0,
        COUNT(*) FILTER (WHERE trip_distance >= 100)                               AS distance_ge_100,
        COUNT(*) FILTER (WHERE "PULocationID" IS NULL)                             AS pu_zone_null,
        COUNT(*) FILTER (WHERE passenger_count IS NULL)                            AS passenger_null,
        COUNT(*) FILTER (WHERE tpep_dropoff_datetime < tpep_pickup_datetime)       AS dropoff_before_pickup,
        COUNT(*) FILTER (
            WHERE fare_amount > 0 AND fare_amount < 500
              AND trip_distance > 0 AND trip_distance < 100
              AND "PULocationID" IS NOT NULL
              AND tpep_pickup_datetime >= DATE '{PERIOD_START}'
              AND tpep_pickup_datetime <  DATE '{PERIOD_END}'
        )                                                                          AS clean_rows
    FROM nyc_taxi_raw
    """)
    dq["pct_clean"] = (dq.clean_rows / dq.total_rows * 100).round(2)
    display(dq.T.rename(columns={0: "value"}))
else:
    print("STUB: сирих поїздок немає, аудит пропущено")

,value
total_rows,6590154
outside_period,17
min_pickup,2002-12-31 22:17:10
max_pickup,2024-04-01 00:34:55
fare_le_0,101255
fare_ge_500,85
distance_le_0,155393
distance_ge_100,193
pu_zone_null,0
passenger_null,611800


**Як читати аудит.** Колонки `*_le_0`, `*_ge_*` і `pu_zone_null` — це кількість рядків, які відкидає кожне правило окремо (правила перетинаються, тому їхня сума більша за кількість відкинутих). `clean_rows` / `pct_clean` — результат усіх правил разом, і саме ці рядки потрапляють у денний ряд.

На що звернути увагу в TLC-файлах:
- **`outside_period > 0`, `min_pickup` / `max_pickup` поза місяцем.** У файлі місяця трапляються записи з датами інших місяців і навіть років. Без фільтра періоду такий запис створив би в денному ряду окремий «день», а `generate_series` spine розтягнувся б на роки.
- **`fare_le_0`.** Від'ємні суми — це повернення та коригування. Вони дають основну частину відкинутих рядків.
- **`passenger_null`.** Пропуски в `passenger_count` не є критерієм відкидання: колонка не бере участі в денних ознаках.
- **`dropoff_before_pickup`.** Окрема аномалія тривалості. Умова її не вимагає, тож рядки лишаються в агрегатах кількості та ціни, але для ознаки тривалості їх довелося б відфільтрувати.

### 1.4. Production-style: `TIMESTAMPTZ` і перехід на літній час

`t7_hw_taxi_trips_typed` інтерпретує локальний NYC час через `AT TIME ZONE 'America/New_York'` і зберігає момент події як `TIMESTAMPTZ`. Різниця між типами має практичний наслідок: у ніч переходу на літній час (2024-03-10, 02:00 → 03:00) **локальний `TIMESTAMP` бреше про тривалість** поїздки на годину, а `TIMESTAMPTZ` — ні. Нижче це перевіряється на даних, а не лише декларується.

In [6]:
if not USE_STUB:
    run_sql(f"""
    CREATE TABLE t7_hw_taxi_trips_typed AS
    SELECT
        "PULocationID"::INTEGER AS pickup_zone_id,
        "DOLocationID"::INTEGER AS dropoff_zone_id,
        tpep_pickup_datetime    AS pickup_local,
        tpep_dropoff_datetime   AS dropoff_local,
        (tpep_pickup_datetime  AT TIME ZONE 'America/New_York') AS pickup_at,
        (tpep_dropoff_datetime AT TIME ZONE 'America/New_York') AS dropoff_at,
        trip_distance::NUMERIC(10, 2) AS trip_distance,
        fare_amount::NUMERIC(10, 2)   AS fare_amount,
        passenger_count::INTEGER      AS passenger_count
    FROM nyc_taxi_raw
    WHERE fare_amount > 0
      AND fare_amount < 500
      AND trip_distance > 0
      AND trip_distance < 100
      AND "PULocationID" IS NOT NULL
      AND tpep_pickup_datetime >= DATE '{PERIOD_START}'
      AND tpep_pickup_datetime <  DATE '{PERIOD_END}';
    """)
    display(q("""
    SELECT pickup_local, pickup_at, pg_typeof(pickup_local) AS local_type, pg_typeof(pickup_at) AS pickup_type
    FROM t7_hw_taxi_trips_typed
    LIMIT 3
    """))
    display(q("""
    SELECT
        COUNT(*) FILTER (WHERE (pickup_at AT TIME ZONE 'America/New_York') <> pickup_local)
            AS nonexistent_local_pickups,
        COUNT(*) FILTER (WHERE (dropoff_at - pickup_at) <> (dropoff_local - pickup_local))
            AS trips_with_wrong_local_duration,
        MIN(pickup_local) FILTER (WHERE (pickup_at AT TIME ZONE 'America/New_York') <> pickup_local)
            AS first_nonexistent_local,
        MAX(pickup_local) FILTER (WHERE (pickup_at AT TIME ZONE 'America/New_York') <> pickup_local)
            AS last_nonexistent_local
    FROM t7_hw_taxi_trips_typed
    """))
    display(q("""
    SELECT pickup_local, dropoff_local,
           dropoff_local - pickup_local AS duration_local_timestamp,
           dropoff_at - pickup_at       AS duration_timestamptz
    FROM t7_hw_taxi_trips_typed
    WHERE (dropoff_at - pickup_at) <> (dropoff_local - pickup_local)
    ORDER BY pickup_local
    LIMIT 5
    """))
else:
    print("STUB: typed-таблицю поїздок пропущено")

,pickup_local,pickup_at,local_type,pickup_type
0,2024-02-01 00:04:45,2024-02-01 05:04:45+00:00,timestamp without time zone,timestamp with time zone
1,2024-02-01 00:56:31,2024-02-01 05:56:31+00:00,timestamp without time zone,timestamp with time zone
2,2024-02-01 00:07:50,2024-02-01 05:07:50+00:00,timestamp without time zone,timestamp with time zone


,nonexistent_local_pickups,trips_with_wrong_local_duration,first_nonexistent_local,last_nonexistent_local
0,0,1044,None,None


,pickup_local,dropoff_local,duration_local_timestamp,duration_timestamptz
0,2024-03-09 03:44:16,2024-03-10 03:36:07,0 days 23:51:51,0 days 22:51:51
1,2024-03-09 07:13:07,2024-03-11 13:55:34,2 days 06:42:27,2 days 05:42:27
2,2024-03-09 10:23:08,2024-03-10 10:11:04,0 days 23:47:56,0 days 22:47:56
3,2024-03-09 10:30:15,2024-03-10 10:28:03,0 days 23:57:48,0 days 22:57:48
4,2024-03-09 11:33:24,2024-03-10 11:01:21,0 days 23:27:57,0 days 22:27:57


**Висновок.** `pg_typeof` показує, що `pickup_local` — `timestamp without time zone` (wall-clock NYC), а `pickup_at` — `timestamp with time zone`, тобто момент на часовій осі. Друга таблиця перевіряє, де ця різниця матеріальна:
- **`nonexistent_local_pickups`.** Локальні часи, яких **не існує** через перехід на літній час (у березні 2024 це 2024-03-10, 02:00–02:59 EST → EDT). PostgreSQL інтерпретує такий час за стандартним зміщенням, тож зворотне перетворення дає іншу «стінну» годину. Ненульове значення означає, що таксометр записав неіснуючий час.
- **`trips_with_wrong_local_duration`.** Поїздки, для яких `dropoff_local − pickup_local` відрізняється від `dropoff_at − pickup_at`. Для поїздки через межу DST локальна різниця завищена або занижена на годину. Тривалість, порахована з `TIMESTAMP`, — тиха помилка в ознаці `trip_duration`.

Для денного агрегату важливий локальний день, тому `ts_day = pickup_local::DATE`, як і вимагає умова.

### 1.5. `t7_hw_taxi_daily`: денний ряд із calendar spine

Таблиця має явний DDL з `CHECK` і PK `(pickup_zone_id, ts_day)`. День — локальний NYC день (`DATE`), як вимагає умова.
- **TLC:** `generate_series` усіх днів періоду × усі зони, що трапились у чистих даних, а потім `LEFT JOIN` денного агрегату. День без поїздок отримує `trip_count = 0`, `total_distance = 0` і `avg_fare = NULL`: середньої ціни немає, і 0 був би хибним значенням.
- **Stub:** DDL і seed із завдання (вже щільний ряд 60 днів × 5 зон).

Для демонстрації в 2.2 зберігається і **розріджений** агрегат без spine (`t7_hw_taxi_daily_sparse`), тобто те, що дає `CREATE TABLE AS ... GROUP BY` із шаблону.

In [7]:
DAILY_DDL = """
CREATE TABLE t7_hw_taxi_daily (
    pickup_zone_id  INTEGER NOT NULL,
    ts_day          DATE    NOT NULL,
    trip_count      INTEGER NOT NULL CHECK (trip_count >= 0),
    avg_fare        NUMERIC(10, 2) CHECK (avg_fare IS NULL OR avg_fare >= 0),
    total_distance  NUMERIC(12, 2) NOT NULL CHECK (total_distance >= 0),
    PRIMARY KEY (pickup_zone_id, ts_day),
    CHECK ((trip_count = 0) = (avg_fare IS NULL))
);
"""

if USE_STUB:
    run_sql("""
    CREATE TABLE t7_hw_taxi_daily (
        pickup_zone_id  INTEGER NOT NULL,
        ts_day          DATE    NOT NULL,
        trip_count      INTEGER NOT NULL CHECK (trip_count >= 0),
        avg_fare        NUMERIC(10, 2) NOT NULL CHECK (avg_fare >= 0),
        total_distance  NUMERIC(12, 2) NOT NULL CHECK (total_distance >= 0),
        PRIMARY KEY (pickup_zone_id, ts_day)
    );

    INSERT INTO t7_hw_taxi_daily (pickup_zone_id, ts_day, trip_count, avg_fare, total_distance)
    SELECT
        z.zone_id,
        DATE '2024-03-31' - (59 - n) AS ts_day,
        500 + (n * 7) + z.zone_id * 50 + (n % 5) * 30 AS trip_count,
        (12.5 + z.zone_id * 0.7 + (n % 4) * 0.4)::NUMERIC(10, 2) AS avg_fare,
        (1500 + n * 25 + z.zone_id * 120)::NUMERIC(12, 2) AS total_distance
    FROM generate_series(0, 59) AS n
    CROSS JOIN (VALUES (1), (2), (3), (4), (5)) AS z(zone_id);

    CREATE TABLE t7_hw_taxi_daily_sparse AS SELECT * FROM t7_hw_taxi_daily;
    """)
else:
    run_sql(DAILY_DDL + f"""
    CREATE TABLE t7_hw_taxi_daily_sparse AS
    SELECT
        pickup_zone_id,
        pickup_local::DATE AS ts_day,
        COUNT(*)::INTEGER AS trip_count,
        AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
        SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
    FROM t7_hw_taxi_trips_typed
    GROUP BY pickup_zone_id, pickup_local::DATE;

    ALTER TABLE t7_hw_taxi_daily_sparse ADD PRIMARY KEY (pickup_zone_id, ts_day);

    INSERT INTO t7_hw_taxi_daily (pickup_zone_id, ts_day, trip_count, avg_fare, total_distance)
    WITH spine_days AS (
        SELECT d::DATE AS ts_day
        FROM generate_series(DATE '{PERIOD_START}', DATE '{PERIOD_END}' - 1, INTERVAL '1 day') AS d
    ),
    zones AS (
        SELECT DISTINCT pickup_zone_id FROM t7_hw_taxi_daily_sparse
    )
    SELECT
        z.pickup_zone_id,
        d.ts_day,
        COALESCE(s.trip_count, 0),
        s.avg_fare,
        COALESCE(s.total_distance, 0)
    FROM zones AS z
    CROSS JOIN spine_days AS d
    LEFT JOIN t7_hw_taxi_daily_sparse AS s
        ON s.pickup_zone_id = z.pickup_zone_id
       AND s.ts_day = d.ts_day;
    """)
run_autocommit("ANALYZE t7_hw_taxi_daily")

q("""
SELECT
    (SELECT COUNT(*) FROM t7_hw_taxi_daily)                          AS daily_rows,
    (SELECT COUNT(*) FROM t7_hw_taxi_daily_sparse)                   AS sparse_rows,
    (SELECT COUNT(DISTINCT pickup_zone_id) FROM t7_hw_taxi_daily)    AS n_zones,
    (SELECT COUNT(DISTINCT ts_day) FROM t7_hw_taxi_daily)            AS n_days,
    (SELECT MIN(ts_day) FROM t7_hw_taxi_daily)                       AS first_day,
    (SELECT MAX(ts_day) FROM t7_hw_taxi_daily)                       AS last_day,
    (SELECT COUNT(*) FROM t7_hw_taxi_daily WHERE trip_count = 0)     AS zero_trip_days,
    (SELECT SUM(trip_count) FROM t7_hw_taxi_daily)                   AS total_trips
""")

,daily_rows,sparse_rows,n_zones,n_days,first_day,last_day,zero_trip_days,total_trips
0,15480,13511,258,60,2024-02-01,2024-03-31,1969,6341305


In [8]:
daily_check = q("""
SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10
""")
assert q("SELECT COUNT(*) AS n FROM t7_hw_taxi_daily").n[0] > 100
# Spine invariant: every zone has exactly the same number of days
assert q("SELECT COUNT(DISTINCT n) AS k FROM (SELECT COUNT(*) AS n FROM t7_hw_taxi_daily GROUP BY pickup_zone_id) AS x").k[0] == 1
daily_check

,pickup_zone_id,n_days
0,1,60
1,2,60
2,3,60
3,4,60
4,6,60
5,7,60
6,8,60
7,9,60
8,10,60
9,11,60


**Висновок.** Таблиця щільна: кожна зона має однакову кількість днів (перевірено `assert`). Для TLC 2024-02 + 2024-03 це **60** днів, бо 2024 рік високосний і лютий має 29 днів. `daily_rows` = `n_zones × 60`, і це значно більше 100. `zero_trip_days` — пари «зона × день», які додав spine: у розрідженому агрегаті (`sparse_rows`) їх просто немає. Найбільше таких днів у периферійних зонах і в службових зонах TLC 264 / 265 (Unknown / Outside of NYC).

## 2. Time-series ML features

Усі вікна мають `PARTITION BY pickup_zone_id`. Чотири запити — як у шаблоні, по 20 рядків, для зони з найбільшою кількістю поїздок. Так ознаки ілюструються на «живому» ряді, а не на зоні, де майже всі дні нульові.

In [9]:
DEMO_ZONE = int(q("""
SELECT pickup_zone_id FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id ORDER BY SUM(trip_count) DESC, pickup_zone_id LIMIT 1
""").pickup_zone_id[0])
print("Демо-зона:", DEMO_ZONE)

Демо-зона: 161


### 2.1. Feature 1: lag-1d / lag-7d / lag-30d

In [10]:
q(f"""
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WHERE pickup_zone_id IN ({DEMO_ZONE})
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,161,2024-02-01,6373,NaN,NaN,None
1,161,2024-02-02,4675,6373.0,NaN,None
2,161,2024-02-03,3605,4675.0,NaN,None
3,161,2024-02-04,2801,3605.0,NaN,None
4,161,2024-02-05,4545,2801.0,NaN,None
5,161,2024-02-06,6115,4545.0,NaN,None
6,161,2024-02-07,6152,6115.0,NaN,None
7,161,2024-02-08,6819,6152.0,6373.0,None
8,161,2024-02-09,5025,6819.0,4675.0,None
9,161,2024-02-10,3745,5025.0,3605.0,None


Фільтр `WHERE pickup_zone_id IN (...)` тут стоїть **до** обчислення вікон, бо `WHERE` виконується раніше за window functions. Для однієї зони це безпечно: її partition і так повний. Фільтрувати за датою **до** `LAG` було б помилкою, бо перші рядки втратили б свою історію. Тому у view нижче фільтр `trips_lag_30d IS NOT NULL` стоїть **зовні** CTE.

### 2.2. Чому потрібен calendar spine: `LAG` рахує рядки, а не дні

На розрідженому агрегаті (без spine) перевіряємо, чи `LAG(…, 7)` справді дивиться на 7 днів назад: порівнюємо `LAG(ts_day, 7)` з `ts_day - 7`. Те саме для вікна 7 днів: `ROWS BETWEEN 6 PRECEDING` проти `RANGE BETWEEN INTERVAL '6 days' PRECEDING`.

In [11]:
LAG_CHECK = """
SELECT
    COUNT(*) FILTER (WHERE lag7_day IS NOT NULL)                                    AS rows_with_lag7,
    COUNT(*) FILTER (WHERE lag7_day IS NOT NULL AND lag7_day <> ts_day - 7)         AS lag7_not_7_days,
    COUNT(*) FILTER (WHERE lag30_day IS NOT NULL AND lag30_day <> ts_day - 30)      AS lag30_not_30_days,
    COUNT(*) FILTER (WHERE rows_window_days <> range_window_days)                   AS rows_vs_range_differ,
    MAX(ts_day - lag7_day)                                                          AS max_real_gap_for_lag7
FROM (
    SELECT
        ts_day,
        LAG(ts_day, 7)  OVER w AS lag7_day,
        LAG(ts_day, 30) OVER w AS lag30_day,
        COUNT(*) OVER (PARTITION BY pickup_zone_id ORDER BY ts_day
                       ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rows_window_days,
        COUNT(*) OVER (PARTITION BY pickup_zone_id ORDER BY ts_day::TIMESTAMP
                       RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW) AS range_window_days
    FROM {table}
    WINDOW w AS (PARTITION BY pickup_zone_id ORDER BY ts_day)
) AS x
"""
spine_demo = pd.concat([
    q(LAG_CHECK.format(table="t7_hw_taxi_daily_sparse")).assign(table="sparse (без spine)"),
    q(LAG_CHECK.format(table="t7_hw_taxi_daily")).assign(table="t7_hw_taxi_daily (spine)"),
]).set_index("table")
assert spine_demo.loc["t7_hw_taxi_daily (spine)", "lag7_not_7_days"] == 0
assert spine_demo.loc["t7_hw_taxi_daily (spine)", "lag30_not_30_days"] == 0
spine_demo

,rows_with_lag7,lag7_not_7_days,lag30_not_30_days,rows_vs_range_differ,max_real_gap_for_lag7
table,,,,,
sparse (без spine),11748,1342,1228,1418,50
t7_hw_taxi_daily (spine),13674,0,0,0,7


**Висновок (перевіряється на ваших даних, рядок `spine` гарантовано нульовий через `assert`).**
- **Рядок `sparse (без spine)`.** `lag7_not_7_days > 0` означає, що в такій кількості рядків «lag-7d» насправді дивиться не на 7 днів назад. `max_real_gap_for_lag7` показує найгірший випадок: на скільки днів назад реально посилається «тижневий» лаг. Для зон із рідкими поїздками це тижні. Ще серйозніше, що такі значення **не `NULL`**, тож модель не має як помітити помилку.
- **`rows_vs_range_differ`.** На розрідженому ряді `ROWS 6 PRECEDING` («7 рядків») і `RANGE INTERVAL '6 days'` («7 календарних днів») дають різні вікна. На щільному ряді вони ідентичні.
- **Рядок `t7_hw_taxi_daily (spine)`** має 0 розбіжностей, а `max_real_gap_for_lag7 = 7`. Ось чому spine — передумова коректних lag-ознак, а не косметика.

У stub-режимі обидва рядки нульові: stub-ряд щільний від початку.

### 2.3. Feature 2: rolling 7-day average fare

`RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW` означає 7 календарних днів включно з поточним. 6, а не 7: інакше вікно охоплювало б 8 днів (off-by-one). `days_in_window` показує фактичну кількість рядків у вікні, `days_with_fare` — скільки з них мали поїздки (`AVG` ігнорує `NULL` у днях без поїздок).

In [12]:
q(f"""
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare)   OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)        OVER w7 AS days_in_window,
    COUNT(avg_fare) OVER w7 AS days_with_fare
FROM t7_hw_taxi_daily
WHERE pickup_zone_id IN ({DEMO_ZONE})
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window,days_with_fare
0,161,2024-02-01,16.91,16.910000,1,1
1,161,2024-02-02,15.44,16.175000,2,2
2,161,2024-02-03,14.65,15.666667,3,3
3,161,2024-02-04,15.91,15.727500,4,4
4,161,2024-02-05,14.28,15.438000,5,5
5,161,2024-02-06,15.03,15.370000,6,6
6,161,2024-02-07,16.10,15.474286,7,7
7,161,2024-02-08,16.68,15.441429,7,7
8,161,2024-02-09,16.15,15.542857,7,7
9,161,2024-02-10,15.56,15.672857,7,7


### 2.4. Feature 3: rolling 30-day fare volatility

In [13]:
q(f"""
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WHERE pickup_zone_id IN ({DEMO_ZONE})
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,161,2024-02-01,16.91,NaN,1
1,161,2024-02-02,15.44,1.039447,2
2,161,2024-02-03,14.65,1.146923,3
3,161,2024-02-04,15.91,0.944330,4
4,161,2024-02-05,14.28,1.043010,5
5,161,2024-02-06,15.03,0.947650,6
6,161,2024-02-07,16.10,0.908017,7
7,161,2024-02-08,16.68,0.942565,8
8,161,2024-02-09,16.15,0.898888,9
9,161,2024-02-10,15.56,0.848377,10


### 2.5. Feature 4: expanding cumulative trips

In [14]:
q(f"""
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WHERE pickup_zone_id IN ({DEMO_ZONE})
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,161,2024-02-01,6373,6373,6373.000000
1,161,2024-02-02,4675,11048,5524.000000
2,161,2024-02-03,3605,14653,4884.333333
3,161,2024-02-04,2801,17454,4363.500000
4,161,2024-02-05,4545,21999,4399.800000
5,161,2024-02-06,6115,28114,4685.666667
6,161,2024-02-07,6152,34266,4895.142857
7,161,2024-02-08,6819,41085,5135.625000
8,161,2024-02-09,5025,46110,5123.333333
9,161,2024-02-10,3745,49855,4985.500000


**Висновок за чотирма групами ознак (демо-зона — найзавантаженіша):**
- **Lag.** `trips_lag_1d` дорівнює `trip_count` попереднього рядка, `trips_lag_7d` на 8-му дні дорівнює значенню 1-го дня. Перші 1 / 7 / 30 рядків мають `NULL`, бо історії ще немає.
- **Rolling 7d.** `days_in_window` росте 1, 2, …, 7 і далі стабільно 7, що підтверджує frame з 7 календарних днів (без off-by-one). `days_with_fare < days_in_window` можливе лише в днях без поїздок: `AVG` ігнорує їхній `NULL`.
- **Rolling 30d volatility.** У першому рядку `NULL`, бо `STDDEV_SAMP` по одному значенню не визначений (ділення на n − 1 = 0). Далі вікно росте до 30 днів.
- **Expanding.** `cumulative_trips` — монотонна сума від початку періоду, `expanding_avg_trips = cumulative_trips / номер дня`.

**Leakage.** Усі вікна включають `CURRENT ROW`, тобто значення того самого дня. Це коректно, якщо прогноз робиться **після** закриття дня на наступні дні (target = `LEAD(trip_count)`). Якщо ж прогнозується `trip_count` того самого дня, вікна мають закінчуватися на `1 PRECEDING`.

### 2.6. Combined feature view `v_t7_hw_taxi_features`

Усі чотири window definitions мають `PARTITION BY pickup_zone_id`. Фільтр `trips_lag_30d IS NOT NULL` стоїть **поза** CTE: спершу ознаки рахуються на повній історії, і лише потім відкидаються рядки без 30-денної історії.

In [15]:
run_sql("""
DROP VIEW IF EXISTS v_t7_hw_taxi_features;

CREATE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;
""")

# Every window definition in the view must contain PARTITION BY pickup_zone_id
view_sql = q("SELECT pg_get_viewdef('v_t7_hw_taxi_features'::regclass, true) AS def")["def"][0]
n_windows = len(re.findall(r"\b(w_pid|w7|w30|we) AS \(", view_sql))
# pg_get_viewdef qualifies columns with the table name, so allow an optional "table." prefix
n_partitioned = len(re.findall(r"AS \(\s*PARTITION BY (?:\w+\.)?pickup_zone_id\b", view_sql))
print(f"Window definitions у view: {n_windows}, з PARTITION BY pickup_zone_id: {n_partitioned}")
assert n_windows == n_partitioned == 4

q(f"""
SELECT *
FROM v_t7_hw_taxi_features
WHERE pickup_zone_id = {DEMO_ZONE}
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

Window definitions у view: 4, з PARTITION BY pickup_zone_id: 4


,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,161,2024-03-02,3896,15.39,8682.43,5307,3979,6373,16.001429,0.801304,152561,4921.322581
1,161,2024-03-03,2872,16.84,8120.44,3896,2735,4675,16.022857,0.823020,155433,4857.281250
2,161,2024-03-04,4930,14.84,11042.26,2872,3997,3605,15.841429,0.814569,160363,4859.484848
3,161,2024-03-05,6655,15.71,14413.87,4930,6178,2801,15.875714,0.814492,167018,4912.294118
4,161,2024-03-06,6629,16.13,14854.09,6655,6543,4545,15.940000,0.764099,173647,4961.342857
5,161,2024-03-07,6710,16.91,16751.37,6629,7285,6115,16.001429,0.770801,180357,5009.916667
6,161,2024-03-08,4974,16.16,12104.05,6710,5307,6152,15.997143,0.771356,185331,5008.945946
7,161,2024-03-09,4152,15.10,8920.21,4974,3896,6819,15.955714,0.771897,189483,4986.394737
8,161,2024-03-10,2775,16.09,7364.84,4152,2872,5025,15.848571,0.771228,192258,4929.692308
9,161,2024-03-11,4954,15.01,11178.10,2775,4930,3745,15.872857,0.785245,197212,4930.300000


In [16]:
q("""
SELECT
    COUNT(*)                          AS feature_rows,
    COUNT(DISTINCT pickup_zone_id)    AS zones,
    MIN(ts_day)                       AS first_feature_day,
    MAX(ts_day)                       AS last_feature_day,
    COUNT(*) / NULLIF(COUNT(DISTINCT pickup_zone_id), 0) AS rows_per_zone,
    COUNT(*) FILTER (WHERE rolling_std_fare_30d IS NULL) AS std_null_rows
FROM v_t7_hw_taxi_features
""")

,feature_rows,zones,first_feature_day,last_feature_day,rows_per_zone,std_null_rows
0,7740,258,2024-03-02,2024-03-31,30,164


### 2.7. Що робить пропущений `PARTITION BY`

Той самий `LAG(trip_count, 1)`, але без `PARTITION BY`. Рахуємо рядки, де «попередній день» насправді належить **іншій зоні**: у першого дня кожної зони, крім найпершої, `LAG` бере останній день попередньої зони.

In [17]:
q("""
SELECT
    COUNT(*) FILTER (WHERE prev_zone IS NOT NULL AND prev_zone <> pickup_zone_id) AS lag_from_other_zone,
    COUNT(DISTINCT pickup_zone_id) - 1                                            AS expected_zone_boundaries,
    COUNT(*) FILTER (WHERE lag_ok IS NULL AND lag_bad IS NOT NULL)                AS silent_non_null_lags
FROM (
    SELECT
        pickup_zone_id,
        LAG(pickup_zone_id) OVER (ORDER BY pickup_zone_id, ts_day)                    AS prev_zone,
        LAG(trip_count, 1)  OVER (ORDER BY pickup_zone_id, ts_day)                    AS lag_bad,
        LAG(trip_count, 1)  OVER (PARTITION BY pickup_zone_id ORDER BY ts_day)        AS lag_ok
    FROM t7_hw_taxi_daily
) AS x
""")

,lag_from_other_zone,expected_zone_boundaries,silent_non_null_lags
0,257,257,257


**Висновок.** Без `PARTITION BY` кількість `lag_from_other_zone` точно дорівнює `кількість зон − 1`. Перший день кожної зони, крім першої, отримує `trips_lag_1d` з останнього дня **попередньої** зони. `silent_non_null_lags` показує, що всі ці значення не `NULL`: правильний запит повернув би тут `NULL` (історії немає), а неправильний — правдоподібне число. Валідація типів цю помилку не зловить, а модель вивчить «історію» чужої сутності. Для `LAG(…, 30)` і rolling-вікон уражена кількість рядків у ~30 разів більша.

## 3. JSONB event-log `t7_hw_events`

Seed — із завдання: 50 000 подій, 5 типів, 1 000 користувачів, детерміновано через `generate_series` (без `RANDOM()`), тож результати між запусками стабільні.

In [18]:
EVENTS_SQL = r"""
DROP TABLE IF EXISTS t7_hw_checkouts_norm;
DROP TABLE IF EXISTS t7_hw_events_write_test;
DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;
"""
run_sql(EVENTS_SQL)
run_autocommit("ANALYZE t7_hw_events")
q("""
SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type
""")

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


Схеми payload за типом подій. Набір ключів знаходиться з самих даних через `jsonb_object_keys`, а не з коду seed:

In [19]:
schemas = q("""
SELECT
    event_type,
    STRING_AGG(DISTINCT k, ', ' ORDER BY k) AS payload_keys,
    COUNT(DISTINCT k) AS n_keys
FROM t7_hw_events
CROSS JOIN LATERAL jsonb_object_keys(payload) AS k
GROUP BY event_type
ORDER BY event_type
""")
schemas

,event_type,payload_keys,n_keys
0,add_to_cart,"price, product_id, quantity, tags",4
1,checkout,"items_count, order_id, payment_method, total",4
2,click,"device, element_id, page_url",3
3,page_view,"device, referrer, url",3
4,recommendation_shown,"experiment, model_name, model_version, recommendations",4


### 3.1. JSONB Query 1: containment `@>`

In [20]:
jq1 = q("""
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
ORDER BY event_id
LIMIT 10
""")
jq1

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


### 3.2. JSONB Query 2: key existence `?` та `?|`

In [21]:
jq2a = q("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
ORDER BY event_id
LIMIT 10
""")
jq2a

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3, 'product_id': 8}"
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity': 3, 'product_id': 13}"
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quantity': 3, 'product_id': 18}"
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3, 'product_id': 23}"
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity': 3, 'product_id': 28}"
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quantity': 3, 'product_id': 33}"
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3, 'product_id': 38}"
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity': 3, 'product_id': 43}"
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quantity': 3, 'product_id': 48}"


In [22]:
jq2b = q("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
ORDER BY event_id
LIMIT 10
""")
jq2b

,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendatio..."
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendatio..."
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendatio..."
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendatio..."
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendatio..."
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendatio..."
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendatio..."
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendatio..."
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendatio..."
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendatio..."


In [23]:
# Key existence is top-level only: a value inside an array or nested object is not a key
q("""
SELECT
    COUNT(*) FILTER (WHERE payload ? 'tags')                         AS has_key_tags,
    COUNT(*) FILTER (WHERE payload ? 'promo')                        AS has_key_promo,
    COUNT(*) FILTER (WHERE payload -> 'tags' ? 'promo')              AS tags_array_contains_promo,
    COUNT(*) FILTER (WHERE payload @> '{"tags": ["promo"]}')         AS containment_tags_promo,
    COUNT(*) FILTER (WHERE payload ?| ARRAY['experiment', 'model_version']) AS has_any_rec_key,
    COUNT(*) FILTER (WHERE payload ?& ARRAY['experiment', 'model_version']) AS has_all_rec_keys
FROM t7_hw_events
""")

,has_key_tags,has_key_promo,tags_array_contains_promo,containment_tags_promo,has_any_rec_key,has_all_rec_keys
0,10000,0,3333,3333,10000,10000


### 3.3. JSONB Query 3: JSONPath

In [24]:
jq3a = q("""
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
ORDER BY event_id
LIMIT 15
""")
jq3a

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [25]:
jq3b = q("""
SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
ORDER BY event_id
LIMIT 10
""")
jq3b

,event_id,order_id,total
0,253,o_100253,303.0
1,258,o_100258,308.0
2,263,o_100263,313.0
3,268,o_100268,318.0
4,273,o_100273,323.0
5,278,o_100278,328.0
6,283,o_100283,333.0
7,288,o_100288,338.0
8,293,o_100293,343.0
9,298,o_100298,348.0


In [26]:
# Cross-check: JSONPath filter must agree with the equivalent cast-and-compare predicate
q("""
SELECT
    COUNT(*) FILTER (WHERE jsonb_path_exists(payload, '$.total ? (@ > 300)')) AS by_jsonpath,
    COUNT(*) FILTER (WHERE (payload ->> 'total')::NUMERIC > 300)               AS by_cast,
    COUNT(*) FILTER (WHERE payload @? '$.total ? (@ > 300)')                   AS by_at_question_operator
FROM t7_hw_events
WHERE event_type = 'checkout'
""")

,by_jsonpath,by_cast,by_at_question_operator
0,5000,5000,5000


**Висновок.**
- **Схеми.** 5 типів по 10 000 подій. З даних (через `jsonb_object_keys`) видно 5 різних схем payload з 3–4 ключами, і спільний ключ `device` мають лише `click` і `page_view`.
- **`@>`** повертає лише checkout-и з `payment_method = card`.
- **`? 'tags'`** знаходить рівно 10 000 подій `add_to_cart`. `?|` з ключами рекомендацій — 10 000 `recommendation_shown`.
- **Top-level семантика `?`.** Ключа `promo` немає в жодному payload (0), хоча значення `"promo"` лежить у масиві `tags` у 3 333 подіях. Знайти його можна через `payload -> 'tags' ? 'promo'` або containment `@> '{"tags": ["promo"]}'`: обидва дають 3 333.
- **JSONPath.** `jsonb_path_query` розгортає масив `recommendations` у 3 рядки на подію. `jsonb_path_exists(... ? (@ > 300))`, оператор `@?` і явний cast `(payload->>'total')::NUMERIC > 300` дають однакові **5 000**. Фільтр JSONPath порівнює числа як числа, без ручного cast.

## 4. GIN-index demo + `EXPLAIN (ANALYZE, BUFFERS)`

### Step 1. Containment `@>` без GIN-індексу

In [27]:
run_sql("""
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;
""")
# VACUUM ANALYZE instead of plain ANALYZE: also sets hint bits and the visibility map after the bulk
# insert, so the first Seq Scan is not penalised and the before/after comparison is fair
run_autocommit("VACUUM ANALYZE t7_hw_events")
# Planner statistics must exist, otherwise row estimates fall back to hard-coded defaults
assert q("SELECT COUNT(*) AS n FROM pg_stats WHERE tablename = 't7_hw_events'").n[0] > 0

In [28]:
card_before_plan, card_before_ms = explain("""
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card')
""")

Aggregate  (cost=1763.09..1763.10 rows=1 width=8) (actual time=57.717..57.719 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=5234 width=0) (actual time=0.014..57.367 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=7
Planning Time: 0.189 ms
Execution Time: 57.744 ms


### Step 2. GIN-індекс `jsonb_path_ops`

In [29]:
run_sql("""
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);
""")
run_autocommit("ANALYZE t7_hw_events")
q("""
SELECT
    pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS gin_index_size,
    pg_size_pretty(pg_relation_size('t7_hw_events'))                      AS table_heap_size
""")

,gin_index_size,table_heap_size
0,1144 kB,8000 kB


### Step 3. Той самий containment після GIN

In [30]:
card_after_plan, card_after_ms = explain("""
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card')
""")

Aggregate  (cost=1091.80..1091.81 rows=1 width=8) (actual time=9.824..9.826 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=34.22..1083.57 rows=3290 width=0) (actual time=0.765..9.490 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..33.40 rows=3290 width=0) (actual time=0.584..0.584 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=16
Planning Time: 0.308 ms
Execution Time: 9.863 ms


In [31]:
google_after_plan, google_after_ms = explain("""
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google')
""")

Aggregate  (cost=1070.18..1070.19 rows=1 width=8) (actual time=5.655..5.657 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=26.06..1063.87 rows=2521 width=0) (actual time=0.839..5.426 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..25.43 rows=2521 width=0) (actual time=0.579..0.580 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.311 ms
Execution Time: 5.707 ms


### Step 3b. `jsonb_path_ops` не обслуговує `?`: потрібен `jsonb_ops`

`jsonb_path_ops` індексує лише хеші шляхів до значень, без окремих ключів, тому оператор key existence `?` ним не прискорюється. Порівнюємо план `payload ? 'tags'` до і після `GIN (payload)` з дефолтним `jsonb_ops`, а також розміри двох GIN-індексів.

In [32]:
tags_pathops_only_plan, tags_pathops_only_ms = explain("""
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ? 'tags'
""")

Aggregate  (cost=1649.87..1649.88 rows=1 width=8) (actual time=10.939..10.941 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=9947 width=0) (actual time=0.011..10.290 rows=10000 loops=1)
        Filter: (payload ? 'tags'::text)
        Rows Removed by Filter: 40000
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=2
Planning Time: 0.146 ms
Execution Time: 10.967 ms


In [33]:
run_sql("CREATE INDEX idx_t7_events_payload_gin_ops ON t7_hw_events USING GIN (payload);")
run_autocommit("ANALYZE t7_hw_events")
tags_ops_plan, tags_ops_ms = explain("""
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ? 'tags'
""")

Aggregate  (cost=1233.06..1233.07 rows=1 width=8) (actual time=5.919..5.921 rows=1 loops=1)
  Buffers: shared hit=1006
  ->  Bitmap Heap Scan on t7_hw_events  (cost=79.09..1207.40 rows=10265 width=0) (actual time=1.043..5.295 rows=10000 loops=1)
        Recheck Cond: (payload ? 'tags'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1006
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_ops  (cost=0.00..76.52 rows=10265 width=0) (actual time=0.836..0.836 rows=10000 loops=1)
              Index Cond: (payload ? 'tags'::text)
              Buffers: shared hit=6
Planning:
  Buffers: shared hit=20
Planning Time: 0.216 ms
Execution Time: 5.973 ms


In [34]:
q("""
SELECT
    pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS gin_jsonb_path_ops,
    pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_ops'))     AS gin_jsonb_ops,
    pg_size_pretty(pg_relation_size('t7_hw_events'))                      AS table_heap
""")

,gin_jsonb_path_ops,gin_jsonb_ops,table_heap
0,1144 kB,1696 kB,8000 kB


### Step 4. Path-specific equality: GIN не допомагає, потрібен expression B-tree

`payload ->> 'device' = 'mobile'` — це рівність **текстового результату виразу**, а не containment. GIN-індекси на `payload` (обидва вже існують) для нього не застосовні.

In [35]:
device_before_plan, device_before_ms = explain("""
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
""")

Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=13.473..13.475 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.018..12.846 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.105 ms
Execution Time: 13.506 ms


In [36]:
run_sql("""
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));
""")
run_autocommit("ANALYZE t7_hw_events")
device_after_plan, device_after_ms = explain("""
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
""")

Aggregate  (cost=1244.06..1244.07 rows=1 width=8) (actual time=1.722..1.723 rows=1 loops=1)
  Buffers: shared hit=2 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=97.29..1223.09 rows=8387 width=0) (actual time=0.509..1.237 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=2 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..95.19 rows=8387 width=0) (actual time=0.363..0.363 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=5 read=1
Planning Time: 0.152 ms
Execution Time: 1.746 ms


Для контексту: той самий логічний фільтр можна записати як containment `payload @> '{"device": "mobile"}'`. Тоді його обслуговує GIN. Але planner підбирає індекс за **формою виразу**, а не за змістом, тож форма запиту має збігатися з формою індексу.

In [37]:
device_containment_plan, device_containment_ms = explain("""
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> '{"device": "mobile"}'
""")

Aggregate  (cost=1215.49..1215.50 rows=1 width=8) (actual time=8.184..8.186 rows=1 loops=1)
  Buffers: shared hit=1006
  ->  Bitmap Heap Scan on t7_hw_events  (cost=74.53..1191.99 rows=9397 width=0) (actual time=1.469..7.501 rows=8333 loops=1)
        Recheck Cond: (payload @> '{"device": "mobile"}'::jsonb)
        Heap Blocks: exact=1000
        Buffers: shared hit=1006
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..72.18 rows=9397 width=0) (actual time=1.270..1.271 rows=8333 loops=1)
              Index Cond: (payload @> '{"device": "mobile"}'::jsonb)
              Buffers: shared hit=6
Planning:
  Buffers: shared hit=12
Planning Time: 0.272 ms
Execution Time: 8.226 ms


In [38]:
gin_summary = pd.DataFrame([
    {"query": "@> payment_method=card", "index": "немає", "ms": card_before_ms, "plan": card_before_plan},
    {"query": "@> payment_method=card", "index": "GIN jsonb_path_ops", "ms": card_after_ms, "plan": card_after_plan},
    {"query": "@> referrer=google", "index": "GIN jsonb_path_ops", "ms": google_after_ms, "plan": google_after_plan},
    {"query": "? 'tags'", "index": "лише jsonb_path_ops", "ms": tags_pathops_only_ms, "plan": tags_pathops_only_plan},
    {"query": "? 'tags'", "index": "+ GIN jsonb_ops", "ms": tags_ops_ms, "plan": tags_ops_plan},
    {"query": "->> 'device' = 'mobile'", "index": "лише GIN", "ms": device_before_ms, "plan": device_before_plan},
    {"query": "->> 'device' = 'mobile'", "index": "+ expression B-tree", "ms": device_after_ms, "plan": device_after_plan},
    {"query": "@> device=mobile", "index": "GIN", "ms": device_containment_ms, "plan": device_containment_plan},
])


def scan_type(plan: str) -> str:
    # The access method actually used to read t7_hw_events
    for kind in ("Index Only Scan", "Bitmap Index Scan", "Index Scan", "Seq Scan"):
        if kind in plan:
            m = re.search(rf"{kind}(?: using (\w+))?", plan)
            return f"{kind} ({m.group(1)})" if m and m.group(1) else kind
    return "?"


gin_summary["access"] = gin_summary.plan.map(scan_type)
gin_summary["shared_hit"] = gin_summary.plan.map(lambda p: int(re.search(r"shared hit=(\d+)", p).group(1)))
gin_summary[["query", "index", "access", "ms", "shared_hit"]]

,query,index,access,ms,shared_hit
0,@> payment_method=card,немає,Seq Scan,57.744,1000
1,@> payment_method=card,GIN jsonb_path_ops,Bitmap Index Scan,9.863,1004
2,@> referrer=google,GIN jsonb_path_ops,Bitmap Index Scan,5.707,1004
3,? 'tags',лише jsonb_path_ops,Seq Scan,10.967,1000
4,? 'tags',+ GIN jsonb_ops,Bitmap Index Scan,5.973,1006
5,->> 'device' = 'mobile',лише GIN,Seq Scan,13.506,1000
6,->> 'device' = 'mobile',+ expression B-tree,Bitmap Index Scan,1.746,2
7,@> device=mobile,GIN,Bitmap Index Scan,8.226,1006


**Висновок за чотирма обов'язковими EXPLAIN (деталі — у Reflection):**

| Запит | До | Після |
|---|---|---|
| `@> payment_method=card` | **Seq Scan**, 46 666 рядків відкинуто фільтром, ~40 мс | **Bitmap Index Scan** на GIN (4 буфери індексу) → **Bitmap Heap Scan** з `Recheck Cond`, ~7 мс |
| `->> 'device' = 'mobile'` | **Seq Scan**, хоча обидва GIN уже існують, ~11 мс | **Bitmap Index Scan** на `idx_t7_events_device_btree`, ~2 мс, **11 буферів** замість 1 000 |

Додатково:
- **`? 'tags'`** з одним `jsonb_path_ops` залишається Seq Scan. Лише `GIN (payload)` з дефолтним `jsonb_ops` перетворює його на Bitmap Index Scan. `jsonb_ops` на ~48 % більший (1 696 проти 1 144 kB), бо індексує окремо ключі й значення.
- **Той самий фільтр `device = mobile`, записаний як `@>`,** обслуговує GIN. Planner обирає індекс за формою виразу.
- **Оцінки рядків** після `VACUUM ANALYZE` близькі до факту (3 612 проти 3 334). Для `->> 'device'` до створення expression index оцінка — дефолтні **250** проти 8 333 реальних, бо статистики для виразу немає. Після `CREATE INDEX` + `ANALYZE` вона стає **8 328**: expression index дає planner ще й статистику виразу.

### Step 5. Ціна індексів на запис

Read/write trade-off вимірюємо прямо: вставляємо ті самі 50 000 подій у порожню копію таблиці без індексів, лише з GIN `jsonb_path_ops` і з повним набором (обидва GIN + expression B-tree). Кожна вставка — свіжа таблиця, медіана з 3 запусків.

In [39]:
import statistics

INSERT_BODY = EVENTS_SQL.split("INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)")[1]
INDEX_SETS = {
    "без індексів": [],
    "GIN jsonb_path_ops": ["CREATE INDEX ON t7_hw_events_write_test USING GIN (payload jsonb_path_ops)"],
    "2× GIN + expression B-tree": [
        "CREATE INDEX ON t7_hw_events_write_test USING GIN (payload jsonb_path_ops)",
        "CREATE INDEX ON t7_hw_events_write_test USING GIN (payload)",
        "CREATE INDEX ON t7_hw_events_write_test ((payload ->> 'device'))",
    ],
}
write_rows = []
for label, ddl in INDEX_SETS.items():
    times = []
    for _ in range(3):
        run_sql("""
        DROP TABLE IF EXISTS t7_hw_events_write_test;
        CREATE TABLE t7_hw_events_write_test (LIKE t7_hw_events INCLUDING DEFAULTS INCLUDING IDENTITY);
        """ + ";\n".join(ddl) + (";" if ddl else ""))
        t0 = time.perf_counter()
        run_sql("INSERT INTO t7_hw_events_write_test (user_id, event_ts, event_type, payload)" + INSERT_BODY)
        times.append(time.perf_counter() - t0)
    size = q("SELECT pg_total_relation_size('t7_hw_events_write_test') AS b").b[0]
    write_rows.append({"indexes": label, "insert_50k_s": round(statistics.median(times), 3),
                       "total_size_mb": round(size / 1e6, 1)})
run_sql("DROP TABLE IF EXISTS t7_hw_events_write_test;")
write_cost = pd.DataFrame(write_rows)
write_cost["slowdown_x"] = (write_cost.insert_50k_s / write_cost.insert_50k_s.iloc[0]).round(2)
write_cost

,indexes,insert_50k_s,total_size_mb,slowdown_x
0,без індексів,0.304,8.2,1.00
1,GIN jsonb_path_ops,0.459,12.3,1.51
2,2× GIN + expression B-tree,0.923,18.5,3.04


**Висновок.** Вставка тих самих 50 000 подій сповільнюється в **~1,5×** з одним GIN `jsonb_path_ops` і в **~3×** з повним набором (2 GIN + expression B-tree). Розмір таблиці з індексами росте з 8,2 до 12,3 і 18,5 MB. Кожна вставка оновлює кожен індекс, а GIN — ще й по кілька записів на один документ (по одному на кожен шлях або ключ payload). Для append-heavy event-log це ціна кожного індексу, яку треба зважувати проти частоти читання.

## 5. JSONB vs normalized

### Step 1. Нормалізована таблиця `t7_hw_checkouts_norm`

In [40]:
run_sql("""
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id, user_id, event_ts, order_id, total, items_count, payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);
""")
run_autocommit("ANALYZE t7_hw_checkouts_norm")
q("""
SELECT
    (SELECT COUNT(*) FROM t7_hw_checkouts_norm) AS n_checkouts,
    pg_size_pretty(pg_relation_size('t7_hw_checkouts_norm')) AS norm_heap_size,
    pg_size_pretty((SELECT SUM(pg_column_size(payload)) FROM t7_hw_events WHERE event_type = 'checkout'))
        AS checkout_payload_bytes_in_jsonb,
    pg_size_pretty((SELECT SUM(pg_column_size(t.*)) FROM t7_hw_checkouts_norm AS t))
        AS checkout_row_bytes_normalized
""")

,n_checkouts,norm_heap_size,checkout_payload_bytes_in_jsonb,checkout_row_bytes_normalized
0,10000,752 kB,1038 kB,648 kB


### Step 2. Один аналітичний запит двома способами

**JSONB-версія:**

In [41]:
agg_jsonb_plan, agg_jsonb_ms = explain("""
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3
""")

Limit  (cost=1953.70..1953.71 rows=3 width=72) (actual time=15.701..15.703 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1953.70..1967.52 rows=5526 width=72) (actual time=15.699..15.700 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1799.39..1882.28 rows=5526 width=72) (actual time=15.663..15.690 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1649.91 rows=9965 width=132) (actual time=0.013..9.941 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=10
Planning Time: 0.164 ms
Execution Time: 15.778 ms


**Normalized-версія:**

In [42]:
agg_norm_plan, agg_norm_ms = explain("""
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3
""")

Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=5.708..5.711 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=5.706..5.708 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=5.693..5.697 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.010..1.118 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning Time: 0.089 ms
Execution Time: 5.744 ms


In [43]:
# Both versions must return identical results
res_jsonb = q("""
SELECT payload ->> 'payment_method' AS payment_method,
       ROUND(AVG((payload ->> 'total')::NUMERIC), 4) AS avg_total, COUNT(*) AS n
FROM t7_hw_events WHERE event_type = 'checkout'
GROUP BY 1 ORDER BY avg_total DESC LIMIT 3
""")
res_norm = q("""
SELECT payment_method, ROUND(AVG(total), 4) AS avg_total, COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY 1 ORDER BY avg_total DESC LIMIT 3
""")
assert res_jsonb.equals(res_norm)
display(res_norm)
pd.DataFrame([
    {"version": "JSONB (t7_hw_events)", "ms": agg_jsonb_ms,
     "shared_hit": int(re.search(r"shared hit=(\d+)", agg_jsonb_plan).group(1)), "access": scan_type(agg_jsonb_plan)},
    {"version": "normalized", "ms": agg_norm_ms,
     "shared_hit": int(re.search(r"shared hit=(\d+)", agg_norm_plan).group(1)), "access": scan_type(agg_norm_plan)},
])

,payment_method,avg_total,n
0,paypal,300.5248,3333
1,card,300.5000,3334
2,crypto,300.4752,3333


,version,ms,shared_hit,access
0,JSONB (t7_hw_events),15.778,1000,Seq Scan
1,normalized,5.744,94,Seq Scan


**Висновок.** Обидві версії повертають ідентичний результат (перевірено `assert`): `paypal` 300,52, `card` 300,50, `crypto` 300,48, по ~3 333 замовлення.
- **JSONB-версія:** Seq Scan по **всіх 50 000** подіях (1 000 буферів, 40 000 відкинуто фільтром `event_type`), а потім для кожного з 10 000 checkout-ів розбір JSONB і `->>`-витяг з cast у `NUMERIC`. ~12 мс.
- **Normalized-версія:** Seq Scan по таблиці, що містить **лише** checkout-и (94 буфери, у ~10,6 раза менше), з уже типізованими колонками. ~3 мс.

Обидва плани використовують `HashAggregate`, тож різницю дає обсяг прочитаних сторінок і вартість витягу з JSONB, а не алгоритм агрегації. Payload checkout-ів займає 1 038 kB, а ті самі дані в typed-рядках — 648 kB: ключі JSONB повторюються в кожному документі.

### Step 3 (bonus). MongoDB-equivalent: pseudocode

```javascript
// MongoDB equivalent: avg total checkout per payment_method, top-3
// Pseudocode — не виконується у PostgreSQL

db.events.aggregate([
  {$match: {event_type: "checkout"}},
  {$group: {
    _id: "$payload.payment_method",
    avg_total: {$avg: {$toDouble: "$payload.total"}},
    n: {$sum: 1}
  }},
  {$sort: {avg_total: -1}},
  {$limit: 3}
]);
```

**Порівняння підходів:**
- **PostgreSQL JSONB** зручний, коли подія має жити поруч із нормалізованими таблицями: `JOIN` з `users` чи `orders`, транзакції (подія і зміна стану атомарно), SQL-аналітика і window functions з розділу 2 над тими самими даними.
- **MongoDB aggregation pipeline** природний для document-first workloads, коли сутність і є документом, схема часто змінюється, а основні запити — вибірка і агрегація всередині однієї колекції без `JOIN`. Мовою запитів є `$match → $group → $sort` замість SQL.
- **High-volume raw clickstream** (мільярди подій, append-only, рідкісні точкові запити) дешевше тримати в document store або data lake (Parquet + partitioning за датою), ніж у рядковому OLTP-сховищі з GIN, ціна запису якого виміряна в Step 5.
- **Curated ML features** зручніше в typed SQL-таблицях: стабільна схема, `CHECK`, `NOT NULL`, контроль типів і передбачувані плани, як у `t7_hw_checkouts_norm` і `t7_hw_taxi_daily`.

## 6. Reflection

**До і після GIN.** До індексу containment `payment_method = card` виконувався як Seq Scan (cost ≈ 1 750, приблизно 40 мс, 46 666 рядків відкинуто фільтром). Після `GIN jsonb_path_ops` план став Bitmap Index Scan → Bitmap Heap Scan (~7 мс). Але буферів було **1 004 проти 1 000**: відповідні події розкидані по всіх 1 000 сторінках (`Heap Blocks: exact=1000`). Виграш дав не I/O, а те, що containment перевіряється на 3 334 кандидатах (`Recheck Cond`), а не на 50 000 рядках.

**Чому для `->> 'device' = 'mobile'` GIN не підходить.** GIN індексує containment і наявність ключів, а не результат виразу `->>`. Planner шукає індекс саме за формою виразу, тож навіть з обома GIN був Seq Scan. Expression B-tree дав Bitmap Index Scan за **11 буферів** замість 1 000. Причина в тому, що B-tree точний і для `COUNT(*)` по all-visible сторінках не читає heap, а `jsonb_path_ops` зберігає хеші й завжди вимагає recheck з читанням рядка.

**Read/write trade-off.** Читання прискорилось у 5–6 разів, але вставка 50 000 подій стала у 1,5 раза повільнішою з одним GIN і у 3 рази — з трьома індексами. Розмір виріс з 8,2 до 18,5 MB.

**JSONB чи нормалізована таблиця.** JSONB — для сирого event-log зі схемою, що змінюється, і для ad-hoc запитів. Нормалізована таблиця — для curated ознак, які читаються постійно: агрегат по checkout-ах прочитав 94 буфери проти 1 000 і виконався у 3–4 рази швидше, ще й з `CHECK`-контролем значень.

**Новий event-type `refund`.** У JSONB достатньо почати писати події: міграції немає, а існуючі GIN-індекси автоматично покривають нові ключі. Потрібні лише валідація payload в застосунку та, за потреби, expression index на новий ключ. У normalized-підході потрібна нова таблиця `t7_hw_refunds_norm` (DDL, `CHECK`, FK на checkout), backfill з event-log і зміна ETL. Це повільніше, зате схема й обмеження явні.